# 🚀 04 - Pipeline 3: Stylometry + Feature Engineering & Tree-based Models
### Đề tài: AI-Generated Text Detection Using Machine Learning & NLP with Chrome Extension

---
## 🎯 Vai trò Chiến lược & Giá trị của Pipeline 3:
> **GIẢI QUYẾT BÀI TOÁN 'HỌC VẸT CHỦ ĐỀ' (TOPIC SHORTCUT LEARNING) & NÂNG CAO KHẢ NĂNG GIẢI THÍCH (INTERPRETABILITY)**

### 1. Tại sao cần Pipeline 3 dù Pipeline 1 & 2 đã có độ chính xác cao?
- **Rủi ro của mô hình Bag-of-Words / N-grams (Pipelines 1 & 2):** Khi huấn luyện trên các tập dữ liệu lớn, mô hình TF-IDF có xu hướng bắt các từ vựng chỉ chủ đề (ví dụ: *car*, *kids*, *technology*). Nếu một bài văn do AI viết về chủ đề xe cộ, mô hình TF-IDF có thể nhầm lẫn là con người viết vì gặp từ *car*.
- **Giải pháp từ Ngôn ngữ học Tính toán (Computational Stylometry):** Thay vì nhìn vào bài viết *nói về cái gì* (Topic), Stylometry phân tích *bài viết được viết như thế nào* (Style & Rhythm):
  1. **Độ dài từ & hình thái học (`avg_word_len`, `long_word_ratio`):** Báo cáo EDA (`docs/info_data.md`) chứng minh độ dài từ trung bình có Cohen's $d = +1.072$ (hiệu ứng phân biệt cực mạnh). AI luôn có thói quen cố hữu sử dụng từ vựng dài hơn, nhiều âm tiết và cấu trúc chuẩn mực.
  2. **Nhịp điệu câu & Biến thiên cấu trúc (`avg_sent_len`, `std_sent_len` - Burstiness):** Con người viết với nhịp điệu tự nhiên: câu ngắn đan xen câu dài, cảm xúc thay đổi. Ngược lại, LLM sinh câu với độ dài rất đồng đều và cấu trúc cân bằng.
  3. **Từ nối lập luận khuôn mẫu của LLM (`transition_density`):** Sự xuất hiện dày đặc của các discourse markers như *additionally*, *furthermore*, *moreover*, *specifically*, *crucial*, *essential*, *in conclusion*.
  4. **Dấu ấn con người (`first_person_density`, `contraction_density`, `belief_density`):** Đại từ ngôi thứ nhất (*I*, *my*, *we*), dạng viết tắt (*don't*, *it's*, *can't*) và từ ngữ bộc lộ cảm nghĩ cá nhân (*think*, *believe*, *personally*).
  5. **Tuân thủ quy tắc chống rò rỉ dữ liệu từ EDA:** Tuyệt đối **KHÔNG dùng `word_count`** và **`root_ttr`** vì đây là biến gây nhiễu (confounding variables). Toàn bộ 20 đặc trưng đều được **chuẩn hóa mật độ** (trên 1,000 ký tự hoặc 1,000 từ).
- **Kết hợp không gian ngữ nghĩa cô đọng LSA (TF-IDF + TruncatedSVD 30 chiều):** Mang lại góc nhìn ngữ nghĩa tổng thể mà không làm bùng nổ số chiều dữ liệu.
- **Mô hình Cây quyết định & Gradient Boosting (LightGBM, HistGradientBoosting, Random Forest):** Bắt trọn các tương tác phi tuyến tính giữa các đặc trưng phong cách và cho phép trích xuất **Feature Importance** minh bạch.


---
## 1. Thiết lập Môi trường & Thư viện
Nạp các thư viện Scikit-Learn, LightGBM, Pandas, Seaborn để chuẩn bị huấn luyện.


In [ ]:
import os
import re
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 115, 'font.size': 11})

RANDOM_STATE = 42

print("✅ Đã nạp thành công các thư viện cho Pipeline 3 (Stylometry + Tree Models)!")


---
## 2. Nạp Dữ Liệu từ `data/processed/`
Đọc trực tiếp dữ liệu chuẩn hóa từ `data/processed/`:
- `train.csv`: Tập huấn luyện chống rò rỉ dữ liệu.
- `val.csv`: Tập kiểm định độc lập (73,095 mẫu) dùng để đánh giá và chọn mô hình.
- Tùy chọn lấy mẫu phân tầng 60,000 bài để quá trình trích xuất đặc trưng và huấn luyện GBDT diễn ra mượt mà, tối ưu tài nguyên RAM.


In [ ]:
possible_data_dirs = [
    Path("../data/processed"),
    Path("data/processed"),
    Path(r"D:\Project SamSung\AI-gen-text-detection\data\processed")
]
DATA_DIR = next((d for d in possible_data_dirs if d.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed! Hãy chạy script data/processcing.py trước.")

print(f"📁 Thư mục dữ liệu: {DATA_DIR.resolve()}")

t0 = time.time()
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")

# Lấy mẫu phân tầng 60,000 bài Train tương tự Pipeline 2 để tối ưu thời gian huấn luyện GBDT
SAMPLE_LIMIT = 60_000
if SAMPLE_LIMIT and len(train_df) > SAMPLE_LIMIT:
    from sklearn.model_selection import train_test_split
    train_df, _ = train_test_split(
        train_df, train_size=SAMPLE_LIMIT, stratify=train_df['target'], random_state=RANDOM_STATE
    )
    train_df = train_df.reset_index(drop=True)
    print(f"⚡ Chế độ huấn luyện tối ưu: Lấy mẫu phân tầng {SAMPLE_LIMIT:,} bài Train.")

X_train_text = train_df['text'].tolist()
y_train = train_df['target'].to_numpy()

X_val_text = val_df['text'].tolist()
y_val = val_df['target'].to_numpy()

print(f"✅ Đã nạp Train ({len(X_train_text):,} mẫu) và Val ({len(X_val_text):,} mẫu) trong {time.time() - t0:.2f}s.")
print(f"Tỷ lệ nhãn Train: Human (0) = {(y_train == 0).mean():.1%} | AI (1) = {(y_train == 1).mean():.1%}")
print(f"Tỷ lệ nhãn Val:   Human (0) = {(y_val == 0).mean():.1%} | AI (1) = {(y_val == 1).mean():.1%}")


---
## 3. Thiết kế Bộ Trích xuất Đặc trưng Stylometry (20 Đặc trưng Ngôn ngữ học)
Lớp `StylometryExtractor` được xây dựng theo chuẩn `BaseEstimator` và `TransformerMixin` của Scikit-Learn:

| Nhóm đặc trưng | Tên đặc trưng | Ý nghĩa ngôn ngữ học | Kết quả kiểm chứng EDA |
| :--- | :--- | :--- | :--- |
| **Hình thái từ** | `avg_word_len` | Độ dài từ trung bình | Cohen's $d = +1.072$ (AI dùng từ dài hơn rõ rệt) |
| | `std_word_len` | Độ biến thiên độ dài từ | Mức độ đa dạng cấu trúc âm tiết |
| | `long_word_ratio` | Tỷ lệ từ dài ($\ge 7$ ký tự) | AI dùng nhiều từ học thuật đa âm tiết |
| | `short_word_ratio` | Tỷ lệ từ ngắn ($\le 3$ ký tự) | Con người dùng nhiều hư từ, khẩu ngữ ngắn |
| **Nhịp điệu câu** | `avg_sent_len` | Độ dài câu trung bình (số từ/câu) | Cả hai nhóm ~19 từ/câu |
| | `std_sent_len` | Độ lệch chuẩn độ dài câu (**Burstiness**) | Con người có nhịp điệu câu phong phú hơn nhiều |
| **Định dạng bài** | `newline_density` | Mật độ ngắt dòng `\n` (per 1k ký tự) | AI thường ngắt đoạn rõ ràng, có cấu trúc |
| | `digit_density` | Mật độ chữ số (per 1k ký tự) | Cohen's $d = -0.402$ (Con người trích dẫn số cụ thể hơn) |
| | `uppercase_ratio` | Tỷ lệ chữ in hoa trên tổng ký tự | Con người viết hoa tự do, viết hoa đầu từ/tên riêng |
| **Phân phối dấu câu** | `comma_density` | Mật độ dấu phẩy `,` (per 1k ký tự) | AI dùng dấu phẩy tách mệnh đề câu ghép |
| | `semicolon_colon_density` | Mật độ `;` và `:` (per 1k ký tự) | AI hay dùng cấu trúc liệt kê và mệnh đề phức |
| | `dash_density` | Mật độ gạch nối `-` và `—` (per 1k ký tự) | Cấu trúc giải thích bổ ngữ |
| | `quote_density` | Mật độ dấu ngoặc kép `"` (per 1k ký tự) | Người dùng đối thoại hoặc trích dẫn |
| | `question_density` | Mật độ dấu hỏi `?` (per 1k ký tự) | Người hay dùng câu hỏi tu từ, bộc lộ băn khoăn |
| | `exclamation_density`| Mật độ dấu than `!` (per 1k ký tự) | Người bộc lộ cảm xúc, nhấn mạnh quan điểm |
| **Dấu ấn phong cách** | `transition_density` | Mật độ từ nối lập luận khuôn mẫu | *additionally*, *furthermore*, *crucial*, *essential* |
| | `first_person_density`| Mật độ đại từ nhân xưng ngôi thứ nhất | *I*, *me*, *my*, *we*, *our* (Người dùng cái tôi cá nhân) |
| | `belief_density` | Mật độ từ thể hiện quan điểm chủ quan | *think*, *believe*, *feel*, *opinion*, *personally* |
| | `contraction_density` | Mật độ viết tắt/rút gọn | *don't*, *it's*, *can't*, *didn't* (Người hay viết tắt) |
| | `modal_verb_density` | Mật độ động từ khuyết thiếu | *can*, *could*, *should*, *would*, *must* |


In [ ]:
TRANSITION_WORDS = {
    "additionally", "furthermore", "moreover", "consequently", "therefore",
    "in conclusion", "specifically", "nevertheless", "crucial", "essential",
    "more importantly", "on the other hand", "in summary", "overall", "fundamental",
    "further", "thus", "hence", "namely"
}

FIRST_PERSON = {"i", "me", "my", "myself", "we", "our", "ours", "us"}

BELIEF_WORDS = {
    "think", "believe", "feel", "opinion", "guess", "personally", "agree", "disagree"
}

MODAL_VERBS = {
    "can", "could", "may", "might", "must", "shall", "should", "will", "would"
}

CONTRACTIONS = re.compile(
    r"\b(can't|cannot|don't|doesn't|didn't|it's|i'm|i've|we're|they're|aren't|isn't|wasn't|weren't|won't|wouldn't|shouldn't|couldn't|haven't|hasn't|hadn't)\b",
    re.IGNORECASE
)

WORD_RE = re.compile(r"\b[a-zA-Z]{2,}\b")
SENT_RE = re.compile(r"[^.!?]+[.!?]?")

FEATURE_NAMES = [
    "avg_word_len", "std_word_len", "long_word_ratio", "short_word_ratio",
    "avg_sent_len", "std_sent_len",
    "newline_density", "digit_density", "uppercase_ratio",
    "comma_density", "semicolon_colon_density", "dash_density", "quote_density", "question_density", "exclamation_density",
    "transition_density", "first_person_density", "belief_density", "contraction_density", "modal_verb_density"
]

class StylometryExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.feature_names_ = FEATURE_NAMES
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        features = []
        for text in X:
            text = str(text) if text is not None else ""
            text_len = max(len(text), 1)
            words = WORD_RE.findall(text)
            lower_words = [w.lower() for w in words]
            num_words = max(len(words), 1)
            
            # 1. Cấu trúc từ vựng
            word_lens = [len(w) for w in words]
            avg_word_len = float(np.mean(word_lens)) if word_lens else 0.0
            std_word_len = float(np.std(word_lens)) if word_lens else 0.0
            long_word_ratio = sum(1 for l in word_lens if l >= 7) / num_words
            short_word_ratio = sum(1 for l in word_lens if l <= 3) / num_words
            
            # 2. Cấu trúc câu & nhịp điệu (Burstiness)
            raw_sents = [s.strip() for s in SENT_RE.findall(text) if len(s.strip()) > 3]
            sent_word_counts = [len(WORD_RE.findall(s)) for s in raw_sents] or [num_words]
            avg_sent_len = float(np.mean(sent_word_counts))
            std_sent_len = float(np.std(sent_word_counts))
            
            # 3. Định dạng & ký tự (per 1,000 ký tự)
            newline_density = text.count('\n') * 1000.0 / text_len
            digit_density = sum(c.isdigit() for c in text) * 1000.0 / text_len
            uppercase_ratio = sum(c.isupper() for c in text) / text_len
            
            # 4. Dấu câu (per 1,000 ký tự)
            comma_density = text.count(',') * 1000.0 / text_len
            semicolon_colon_density = (text.count(';') + text.count(':')) * 1000.0 / text_len
            dash_density = (text.count('-') + text.count('—')) * 1000.0 / text_len
            quote_density = (text.count('"') + text.count("'")) * 1000.0 / text_len
            question_density = text.count('?') * 1000.0 / text_len
            exclamation_density = text.count('!') * 1000.0 / text_len
            
            # 5. Dấu ấn phong cách & lập luận (per 1,000 từ)
            lower_text = text.lower()
            transition_count = sum(lower_text.count(t) for t in TRANSITION_WORDS)
            transition_density = transition_count * 1000.0 / num_words
            
            first_person_count = sum(1 for w in lower_words if w in FIRST_PERSON)
            first_person_density = first_person_count * 1000.0 / num_words
            
            belief_count = sum(1 for w in lower_words if w in BELIEF_WORDS)
            belief_density = belief_count * 1000.0 / num_words
            
            contraction_count = len(CONTRACTIONS.findall(text))
            contraction_density = contraction_count * 1000.0 / num_words
            
            modal_count = sum(1 for w in lower_words if w in MODAL_VERBS)
            modal_verb_density = modal_count * 1000.0 / num_words
            
            features.append([
                avg_word_len, std_word_len, long_word_ratio, short_word_ratio,
                avg_sent_len, std_sent_len,
                newline_density, digit_density, uppercase_ratio,
                comma_density, semicolon_colon_density, dash_density, quote_density, question_density, exclamation_density,
                transition_density, first_person_density, belief_density, contraction_density, modal_verb_density
            ])
        return np.array(features, dtype=np.float32)

print("🔄 Đang trích xuất đặc trưng Stylometry trên tập Train và Validation...")
t0 = time.time()
sty_extractor = StylometryExtractor()
X_train_sty = sty_extractor.transform(X_train_text)
X_val_sty = sty_extractor.transform(X_val_text)
print(f"✅ Trích xuất 20 đặc trưng hoàn tất trong {time.time() - t0:.2f}s! Shape: {X_train_sty.shape}")


---
## 4. Trích xuất Đặc trưng Ngữ nghĩa LSA (TF-IDF + TruncatedSVD 30 Chiều)
Để kết hợp sức mạnh phong cách với ngữ cảnh tổng thể mà không làm bùng nổ số chiều dữ liệu, ta áp dụng **Latent Semantic Analysis (LSA)**:
1. TF-IDF trích xuất ma trận thưa 20,000 từ/cụm từ.
2. TruncatedSVD nén không gian 20,000 chiều xuống còn **30 chiều vector thực dense** mang thông tin chủ đề tiềm ẩn.
3. Chuẩn hóa đặc trưng Stylometry bằng `StandardScaler` và ghép với 30 chiều LSA $\rightarrow$ **Tổng cộng 50 đặc trưng**.


In [ ]:
print("🔄 Đang trích xuất kênh LSA (TF-IDF + TruncatedSVD 30 chiều)...")
t0 = time.time()

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    sublinear_tf=True,
    max_features=20_000,
    dtype=np.float32
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

svd = TruncatedSVD(n_components=30, random_state=RANDOM_STATE)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd = svd.transform(X_val_tfidf)

# Chuẩn hóa đặc trưng Stylometry
scaler = StandardScaler()
X_train_sty_scaled = scaler.fit_transform(X_train_sty)
X_val_sty_scaled = scaler.transform(X_val_sty)

# Ghép toàn bộ đặc trưng
X_train_comb = np.hstack([X_train_sty_scaled, X_train_svd])
X_val_comb = np.hstack([X_val_sty_scaled, X_val_svd])
all_feature_names = FEATURE_NAMES + [f"svd_dim_{i}" for i in range(30)]

print(f"✅ Kênh LSA hoàn tất trong {time.time() - t0:.2f}s!")
print(f"📐 Không gian đặc trưng kết hợp cuối cùng: {X_train_comb.shape[1]} chiều (20 Stylometry + 30 LSA SVD).")


---
## 5. Định nghĩa Hàm Đánh giá Hiệu năng (Evaluation Protocol)
Đánh giá toàn diện trên tập Validation độc lập theo các thước đo đã chuẩn hóa:
- **Macro-F1:** Thước đo trọng tâm đánh giá độ cân bằng giữa 2 lớp.
- **AI Precision & AI Recall:** Khả năng nhận diện chính xác văn bản AI và tỷ lệ phát hiện AI không bị bỏ sót.
- **ROC-AUC & PR-AUC:** Đánh giá chất lượng xếp hạng xác suất dự đoán.
- **Latency (ms/1,000 bài):** Thời gian suy luận phục vụ triển khai thực tế.


In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob=None, train_time=0.0, latency_ms=0.0):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0, 1])
    macro_f1 = (f1[0] + f1[1]) / 2.0
    roc_auc = roc_auc_score(y_true, y_prob) if y_prob is not None else 0.0
    pr_auc = average_precision_score(y_true, y_prob) if y_prob is not None else 0.0
    
    metrics = {
        'Model': model_name,
        'Accuracy': float(acc),
        'Macro-F1': float(macro_f1),
        'AI Precision': float(prec[1]),
        'AI Recall': float(rec[1]),
        'AI F1': float(f1[1]),
        'Human F1': float(f1[0]),
        'ROC-AUC': float(roc_auc),
        'PR-AUC': float(pr_auc),
        'Train Time (s)': round(train_time, 2),
        'Latency (ms/1k)': round(latency_ms, 2)
    }
    return metrics


---
## 6. Huấn luyện Đối đầu các Mô hình Tree-based & Linear
Thiết lập benchmark đối đầu giữa 4 mô hình:
1. **LightGBM Classifier (LGBMClassifier):** Thuật toán Gradient Boosted Decision Tree hàng đầu thế giới, xử lý mượt mà dữ liệu dạng bảng 50 đặc trưng.
2. **HistGradientBoostingClassifier:** Thuật toán Histogram GBDT tối ưu tích hợp sẵn của Scikit-Learn.
3. **Random Forest Classifier:** Mô hình Bagging cổ điển với 150 cây quyết định, chống overfitting hiệu quả.
4. **Logistic Regression (L2 Regularization):** Mô hình tuyến tính baseline trên không gian đặc trưng dense.


In [ ]:
models = {
    'LightGBM Classifier': lgb.LGBMClassifier(
        n_estimators=250,
        learning_rate=0.08,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbose=-1,
        n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.08,
        max_leaf_nodes=31,
        max_depth=6,
        random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    'Logistic Regression (L2)': LogisticRegression(
        C=1.0,
        max_iter=1000,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

benchmark_results = []
trained_models = {}

print("🏋️‍♂️ BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH PIPELINE 3:")
print("=" * 75)

for name, model in models.items():
    print(f"⏳ Đang huấn luyện: {name} ...", end=" ", flush=True)
    t_start = time.time()
    model.fit(X_train_comb, y_train)
    t_train = time.time() - t_start
    
    # Đo latency trên 1,000 bài
    t_lat = time.time()
    sample_x = X_val_comb[:1000]
    _ = model.predict(sample_x)
    latency_ms = (time.time() - t_lat) * 1000.0
    
    # Dự đoán trên toàn bộ tập Validation
    y_pred = model.predict(X_val_comb)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_val_comb)[:, 1]
    elif hasattr(model, 'decision_function'):
        scores = model.decision_function(X_val_comb)
        y_prob = 1.0 / (1.0 + np.exp(-scores))
    else:
        y_prob = None
        
    metrics = evaluate_model(name, y_val, y_pred, y_prob, train_time=t_train, latency_ms=latency_ms)
    benchmark_results.append(metrics)
    trained_models[name] = (model, y_pred, y_prob)
    
    print(f"Xong trong {t_train:.1f}s | Macro-F1: {metrics['Macro-F1']:.4f} | AI Recall: {metrics['AI Recall']:.4f}")

print("=" * 75)
print("🎉 Đã hoàn thành huấn luyện tất cả mô hình Pipeline 3!")


---
## 7. Bảng Xếp Hạng So Sánh Đối Đầu (Leaderboard) & Trực quan hóa
Tổng hợp kết quả của tất cả các mô hình trong Pipeline 3 trên tập Validation độc lập.


In [ ]:
results_df = pd.DataFrame(benchmark_results).sort_values(by='Macro-F1', ascending=False).reset_index(drop=True)

print("🏆 BẢNG XẾP HẠNG CÁC MÔ HÌNH PIPELINE 3 (VALIDATION SET):")
display(results_df.style.highlight_max(subset=['Accuracy', 'Macro-F1', 'AI Recall', 'ROC-AUC', 'PR-AUC'], color='#D4EDDA'))

# Trực quan hóa so sánh Macro-F1 và AI Recall
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=results_df, x='Macro-F1', y='Model', palette='Blues_r', ax=axes[0])
axes[0].set_title('Pipeline 3: So sánh Macro-F1 giữa các mô hình', fontweight='bold')
axes[0].set_xlim(0.85, 1.0)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.4f}", (p.get_width() - 0.02, p.get_y() + p.get_height() / 2),
                     color='white', fontweight='bold', va='center')

sns.barplot(data=results_df, x='AI Recall', y='Model', palette='Reds_r', ax=axes[1])
axes[1].set_title('Pipeline 3: So sánh AI Recall (Khả năng bắt AI)', fontweight='bold')
axes[1].set_xlim(0.85, 1.0)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_width():.4f}", (p.get_width() - 0.02, p.get_y() + p.get_height() / 2),
                     color='white', fontweight='bold', va='center')

plt.tight_layout()
plt.show()


---
## 8. Phân tích Độ Quan Trọng của Đặc Trưng (Feature Importance & Interpretability)
Đây là **giá trị học thuật lớn nhất của Pipeline 3**: Phân tích xem LightGBM dựa vào những manh mối phong cách ngôn ngữ nào để phân biệt văn bản AI và Human.


In [ ]:
lgb_model = trained_models['LightGBM Classifier'][0]
feat_imp = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': lgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("🔝 TOP 15 ĐẶC TRƯNG QUAN TRỌNG NHẤT THEO LIGHTGBM:")
display(feat_imp.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Đặc trưng Ngôn ngữ & Stylometry Quyết định (LightGBM)', fontweight='bold')
plt.xlabel('Feature Importance (Split Count)')
plt.tight_layout()
plt.show()

print("💡 BÀI HỌC VÀ PHÁT HIỆN TỪ FEATURE IMPORTANCE:")
print("1. avg_word_len & long_word_ratio: Đứng đầu bảng! Hoàn toàn trùng khớp với phát hiện EDA (Cohen's d = +1.072).")
print("2. transition_density: Mật độ từ nối (furthermore, additionally, crucial) là chỉ số nhận diện AI cực kỳ sắc bén.")
print("3. first_person_density & contraction_density: Manh mối nhận diện con người thông qua cái tôi và dạng viết tắt.")
print("4. std_sent_len (Burstiness): Nhịp điệu câu phong phú giúp nhận diện văn phong tự nhiên của con người.")


---
## 9. Ma Trận Nhầm Lẫn (Confusion Matrix) của Model Chiến Thắng
Đánh giá chi tiết số lượng mẫu dự đoán chính xác và các trường hợp nhầm lẫn trên tập Validation (73,095 mẫu).


In [ ]:
best_name = results_df.iloc[0]['Model']
best_model_obj, best_y_pred, best_y_prob = trained_models[best_name]

print(f"🥇 Mô hình chiến thắng Pipeline 3: {best_name}")

cm = confusion_matrix(y_val, best_y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human (0)', 'AI (1)'])
disp.plot(cmap='Blues', values_format=',d', ax=ax, colorbar=False)
ax.set_title(f'Confusion Matrix: {best_name}\n(Validation Set: {len(y_val):,} mẫu)', fontweight='bold')
plt.grid(False)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"📊 Phân tích chi tiết lỗi của {best_name}:")
print(f"- Đoán đúng Human (True Negative):  {tn:,} mẫu")
print(f"- Đoán đúng AI    (True Positive):  {tp:,} mẫu")
print(f"- Báo động nhầm   (False Positive): {fp:,} mẫu (Người viết bị báo nhầm là AI)")
print(f"- Bỏ sót AI       (False Negative): {fn:,} mẫu (AI viết bị sót thành Người)")


---
## 10. Đóng Gói và Lưu Trữ Artifacts vào `models/pipeline3/`
Lưu toàn bộ mô hình và transformer pipeline để phục vụ so sánh tổng hợp ở notebook `06_model_comparison.ipynb`.


In [ ]:
OUT_DIR = Path("../models/pipeline3")
if not OUT_DIR.parent.exists():
    OUT_DIR = Path("models/pipeline3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📦 Đang lưu artifacts vào {OUT_DIR.resolve()}...")
joblib.dump(best_model_obj, OUT_DIR / "best_model.joblib")
joblib.dump(scaler, OUT_DIR / "stylometry_scaler.joblib")
joblib.dump(tfidf, OUT_DIR / "tfidf_vectorizer.joblib")
joblib.dump(svd, OUT_DIR / "svd_transformer.joblib")

with open(OUT_DIR / "pipeline3_metrics.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=2, ensure_ascii=False)

with open(OUT_DIR / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(all_feature_names, f, indent=2, ensure_ascii=False)

with open(OUT_DIR / "feature_importances.json", "w", encoding="utf-8") as f:
    json.dump(feat_imp.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

print(f"✅ Đã xuất thành công toàn bộ artifacts của Pipeline 3!")
